This is standalone validation notebook that takes in a `submission.zip` and returns the same `submission.zip` along with evaluation metrics.

These validation concepts were taken from Kh0a's [notebook](https://www.kaggle.com/code/llkh0a/nemotron-unsloth-sft-training-3-30-2).

I want a notebook that does only validation, and prints a few more metrics that I care about.

In [1]:
SUBMISSION_ZIP_PATH = (
    "/kaggle/input/notebooks/huikang/tinker-submission-notebook/submission.zip"
)
RUN_EVALUATION = True
EVALUATION_SAMPLE_SIZE = 950

In [2]:
import zipfile

with zipfile.ZipFile(SUBMISSION_ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall()

# Print configs

In [3]:
import json

with open("adapter_config.json") as f:
    trained_adapter_config = json.load(f)

print(trained_adapter_config)

{'alpha_pattern': {}, 'auto_mapping': None, 'base_model_name_or_path': None, 'bias': 'none', 'corda_config': None, 'eva_config': None, 'exclude_modules': None, 'fan_in_fan_out': False, 'inference_mode': False, 'init_lora_weights': True, 'layer_replication': None, 'layers_pattern': None, 'layers_to_transform': None, 'loftq_config': {}, 'lora_alpha': 32, 'lora_bias': False, 'lora_dropout': 0, 'megatron_config': None, 'megatron_core': 'megatron.core', 'modules_to_save': None, 'peft_type': 'LORA', 'r': 32, 'rank_pattern': {}, 'revision': None, 'target_modules': ['k_proj', 'o_proj', 'in_proj', 'q_proj', 'up_proj', 'v_proj', 'down_proj', 'out_proj', 'lm_head'], 'task_type': 'CAUSAL_LM', 'trainable_token_indices': None, 'use_dora': False, 'use_rslora': False}


# Load model

In [4]:
"""Metric for NVIDIA (129716)."""

import subprocess
import sys

# Set up environment
commands = [
    "uv pip uninstall torch torchvision torchaudio",
    "tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell",
]
if RUN_EVALUATION:
    for cmd in commands:
        print(f"Running: {cmd}")
        subprocess.run(cmd, shell=True, check=True)
sys.path.insert(0, "/tmp")

Running: uv pip uninstall torch torchvision torchaudio


Using Python 3.12.12 environment at: /usr
Uninstalled 3 packages in 781ms
 - torch==2.9.0+cu126
 - torchaudio==2.9.0+cu126
 - torchvision==0.24.0+cu126


Running: tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell


In [5]:
import glob
import math
import multiprocessing
import os
import re
import time
from pathlib import Path

import kagglehub
import pandas as pd
from tqdm import tqdm

# Configuration
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)

In [6]:
class ParticipantVisibleError(Exception):
    pass


def cache_model(
    path: str | Path,
    exts: tuple[str, ...] = (".bin", ".pt", ".safetensors"),
    num_workers: int | None = None,
    chunk_mb: int = 256,
) -> int:
    """Pre-read model weight files into the OS page cache to speed up later loads.

    Args:
        path        : Directory containing model files, or a single file path.
        exts        : File extensions treated as model weight files.
        num_workers : Number of threads (default = min(CPU cores, 8)).
        chunk_mb    : Size of each read chunk in MB.

    Returns:
        Total bytes read (int).
    """
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def warmup_file(fpath: Path) -> tuple[Path, int]:
        """Sequentially read an entire file in chunks."""
        chunk_size = chunk_mb * 1024 * 1024
        total = 0
        try:
            with open(fpath, "rb") as f:
                while True:
                    data = f.read(chunk_size)
                    if not data:
                        break
                    total += len(data)
        except Exception as e:
            print(f"Error reading {fpath}: {e}")
        return fpath, total

    path = Path(path)
    # Collect files to read
    files: list[Path] = []
    if path.is_dir():
        files = [p for p in path.rglob("*") if p.is_file() and str(p).endswith(exts)]
        files.sort()
    else:
        files = [path] if path.exists() else []

    if not files:
        print(f"No model files found to cache at: {path}")
        return 0

    # Decide number of worker threads
    if num_workers is None:
        try:
            num_workers = min(multiprocessing.cpu_count(), 8)
        except Exception:
            num_workers = 4

    print(f"[cache_model] {len(files)} file(s), {num_workers} worker(s)")
    t0 = time.time()
    total_bytes = 0
    # Read files in parallel
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futures = {pool.submit(warmup_file, f): f for f in files}
        for i, fut in enumerate(as_completed(futures), 1):
            fpath, n = fut.result()
            total_bytes += n
            print(f"[{i}/{len(files)}] cached {fpath.name}")

    elapsed = time.time() - t0
    gb = total_bytes / 1024**3
    speed = gb / elapsed if elapsed > 0 else 0
    print(f"[cache_model] total read ≈ {gb:.2f} GB")
    print(f"[cache_model] elapsed {elapsed:.2f} s, ~{speed:.2f} GB/s")
    return total_bytes


def extract_final_answer(text: str | None) -> str:
    r"""Extracts the final answer from the model response.

    Prioritizes extracting answers inside `\boxed{}`.
    If no `\boxed{}` format is found, attempts to extract numbers from other formats.

    Examples:
        >>> extract_final_answer(r"The answer is \boxed{42}")
        '42'
        >>> extract_final_answer("The final answer is: 3.14")
        '3.14'
        >>> extract_final_answer("Just a number 100 in text")
        '100'
        >>> extract_final_answer(None)
        'NOT_FOUND'
    """
    if text is None:
        return "NOT_FOUND"

    # Search for boxed answer
    # Match all instances of \boxed{...} or unclosed \boxed{ at the end
    matches = re.findall(r"\\boxed\{([^}]*)(?:\}|$)", text)
    if matches:
        non_empty = [m.strip() for m in matches if m.strip()]
        if non_empty:
            return non_empty[-1]
        return matches[-1].strip()

    # Other common formats if \boxed{} is not found
    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:：]\s*([^\n]+)",
        r"final answer\s*[:：]\s*([^\n]+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()

    # If no structured format is found, extract the last valid number in the text
    matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if matches:
        return matches[-1]

    # If no numeric answer is found, return the last line of text as a fallback
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else "NOT_FOUND"


def verify(stored_answer: str, predicted: str) -> bool:
    """Verify if the answer matches.

    For numerical answers, allow them to be judged as equal within a certain relative tolerance (1e-2);
    otherwise, compare strictly as strings (case-insensitive).
    """
    # Clean up strings
    stored_answer = stored_answer.strip()
    predicted = predicted.strip()

    try:
        # Try to convert the answers to floating point numbers
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        # Use a small absolute tolerance for numbers near zero
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        # Fallback to case-insensitive string comparison
        return predicted.lower() == stored_answer.lower()



def generate_predictions(
    test_df: pd.DataFrame,
    lora_path: str,
    row_id_col: str,
    max_lora_rank: int,
    max_tokens: int,
    top_p: float,
    temperature: float,
    max_num_seqs: int,
    gpu_memory_utilization: float,
    max_model_len: int,
    debug: bool = False,
) -> pd.DataFrame:
    """Load the model and generate predictions for the provided test data.

    Args:
        debug: If True, writes a CSV file with raw model outputs and extracted predictions.
    """
    # Cache Model
    cache_model(MODEL_PATH, num_workers=16, chunk_mb=1024)

    os.environ["TRANSFORMERS_NO_TF"] = "1"
    os.environ["TRANSFORMERS_NO_FLAX"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest

    # Initialize vLLM Offline inference Engine
    llm = LLM(
        model=str(MODEL_PATH),
        tensor_parallel_size=1,
        max_num_seqs=max_num_seqs,
        gpu_memory_utilization=gpu_memory_utilization,
        dtype="auto",
        max_model_len=max_model_len,
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=max_lora_rank,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
    )

    sampling_params = SamplingParams(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )

    tokenizer = llm.get_tokenizer()
    prompts = []
    for item in test_df.itertuples(index=False):
        user_content = (
            item.prompt
            + "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"
        )
        # Format using the tokenizer's chat template directly
        try:
            prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_content}],
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=True,
            )
        except Exception:
            # Fallback if chat template fails
            prompt = user_content
        prompts.append(prompt)

    # Generate predictions using continuous batching
    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=LoRARequest("adapter", 1, lora_path),
    )

    predictions = []
    debug_records = []
    for item, output in zip(test_df.itertuples(index=False), outputs):
        raw_text = output.outputs[0].text
        extracted_answer = extract_final_answer(raw_text)

        row_id_val = getattr(item, row_id_col)

        predictions.append(
            {
                row_id_col: row_id_val,
                "prediction": extracted_answer,
            }
        )

        if debug:
            debug_records.append(
                {
                    row_id_col: row_id_val,
                    "raw_output": raw_text,
                    "extracted_prediction": extracted_answer,
                }
            )

    # Write debug CSV if requested
    if debug and debug_records:
        debug_df = pd.DataFrame(debug_records)
        debug_df.to_csv("debug_predictions.csv", index=False)
        print("Debug data saved to debug_predictions.csv")

    return pd.DataFrame(predictions)

In [7]:
# Cache Model
if RUN_EVALUATION:
    cache_model(MODEL_PATH, num_workers=16, chunk_mb=1024)

[cache_model] 13 file(s), 16 worker(s)
[1/13] cached model-00013-of-00013.safetensors
[2/13] cached model-00008-of-00013.safetensors
[3/13] cached model-00004-of-00013.safetensors
[4/13] cached model-00005-of-00013.safetensors
[5/13] cached model-00009-of-00013.safetensors
[6/13] cached model-00012-of-00013.safetensors
[7/13] cached model-00001-of-00013.safetensors
[8/13] cached model-00010-of-00013.safetensors
[9/13] cached model-00002-of-00013.safetensors
[10/13] cached model-00011-of-00013.safetensors
[11/13] cached model-00006-of-00013.safetensors
[12/13] cached model-00007-of-00013.safetensors
[13/13] cached model-00003-of-00013.safetensors
[cache_model] total read ≈ 58.82 GB
[cache_model] elapsed 61.32 s, ~0.96 GB/s


# Init vLLM

In [8]:
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

2026-04-12 19:12:29.973449: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776021150.156999      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776021150.213813      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776021150.693904      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776021150.693915      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776021150.693916      65 computation_placer.cc:177] computation placer alr

In [9]:
# www.kaggle.com/competitions/nvidia-nemotron-model-reasoning-challenge/overview/evaluation
max_model_len = 8192
max_lora_rank = 32
max_tokens = 7680
top_p = 1.0
temperature = 0.0
max_num_seqs = 64
gpu_memory_utilization = 0.85
max_model_len = 8192

In [10]:
# Initialize vLLM Offline inference Engine

if RUN_EVALUATION:
    llm = LLM(
        model=str(MODEL_PATH),
        tensor_parallel_size=1,
        max_num_seqs=max_num_seqs,
        gpu_memory_utilization=gpu_memory_utilization,
        dtype="auto",
        max_model_len=max_model_len,
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=max_lora_rank,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
    )

    sampling_params = SamplingParams(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        logprobs=1,
    )

INFO 04-12 19:13:00 [utils.py:238] non-default args: {'trust_remote_code': True, 'max_model_len': 8192, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.85, 'max_num_seqs': 64, 'disable_log_stats': True, 'enable_lora': True, 'max_lora_rank': 32, 'enable_chunked_prefill': True, 'model': '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 04-12 19:13:33 [model.py:531] Resolved architecture: NemotronHForCausalLM
INFO 04-12 19:13:33 [model.py:1554] Using max model len 8192
INFO 04-12 19:13:33 [scheduler.py:231] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-12 19:13:33 [config.py:618] Updating mamba_ssm_cache_dtype to 'float32' for NemotronH model
WARNING 04-12 19:13:33 [config.py:381] Mamba cache mode is set to 'all' for NemotronHForCausalLM by default when prefix caching is enabled
INFO 04-12 19:13:33 [config.py:401] Warning: Prefix caching in Mamba cache 'all' mode is currently enabled. Its support for Mamba layers is experimental. Please report any issues you may observe.
INFO 04-12 19:13:33 [config.py:544] Setting attention block size to 2176 tokens to ensure that attention page size is >= mamba page size.
INFO 04-12 19:13:33 [config.py:575] Padding mamba page size by 4.41% to ensure that mamba page size and attention page size are exactly equal.
INFO 04-12 19:13:33 [vllm.py:747] Asynchron

2026-04-12 19:13:39.629111: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776021219.639764     431 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776021219.642984     431 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776021219.650895     431 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776021219.650913     431 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776021219.650914     431 computation_placer.cc:177] computation placer alr

(EngineCore_DP0 pid=431) INFO 04-12 19:13:51 [core.py:101] Initializing a V1 LLM engine (v0.17.1) with config: model='/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1', speculative_config=None, tokenizer='/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityCon

...

(EngineCore_DP0 pid=431) WARNING 04-12 19:14:10 [fused_moe.py:1093] Using default MoE config. Performance might be sub-optimal! Config file not found at /tmp/vllm/model_executor/layers/fused_moe/configs/E=128,N=1856,device_name=NVIDIA_RTX_PRO_6000_Blackwell_Server_Edition.json
(EngineCore_DP0 pid=431) INFO 04-12 19:14:15 [backends.py:366] Compiling a graph for compile range (1, 16384) takes 10.52 s
(EngineCore_DP0 pid=431) INFO 04-12 19:14:15 [monitor.py:35] torch.compile takes 14.08 s in total
(EngineCore_DP0 pid=431) INFO 04-12 19:14:15 [decorators.py:580] saving AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/a745ba0e914f72dd8b769997a9d9c6e0e60091cdac8cb4325b93233c9af338a2/rank_0_0/model
(EngineCore_DP0 pid=431) INFO 04-12 19:14:16 [decorators.py:588] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/a745ba0e914f72dd8b769997a9d9c6e0e60091cdac8cb4325b93233c9af338a2/rank_0_0/model
(EngineCore_DP0 pid=431) INFO 04-12

(EngineCore_DP0 pid=431) 2026-04-12 19:14:21,336 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore_DP0 pid=431) 2026-04-12 19:14:21,544 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE):   0%|          | 0/38 [00:00<?, ?it/s]

(EngineCore_DP0 pid=431) WARNING 04-12 19:14:22 [utils.py:268] Using default LoRA kernel configs


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 38/38 [00:12<00:00,  3.15it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 22/22 [00:41<00:00,  1.89s/it]


(EngineCore_DP0 pid=431) INFO 04-12 19:15:16 [gpu_model_runner.py:5386] Graph capturing finished in 55 secs, took -3.62 GiB
(EngineCore_DP0 pid=431) INFO 04-12 19:15:16 [core.py:282] init engine (profile, create kv cache, warmup model) took 75.44 seconds
(EngineCore_DP0 pid=431) INFO 04-12 19:15:17 [vllm.py:747] Asynchronous scheduling is enabled.
INFO 04-12 19:15:17 [llm.py:388] Supported tasks: ['generate']


# Test generation

In [11]:
import pandas as pd

df = pd.read_csv("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv")
df = df.head(EVALUATION_SAMPLE_SIZE).copy()

In [12]:
problem_texts = list(df["prompt"])

if RUN_EVALUATION:
    tokenizer = llm.get_tokenizer()
    prompts = []
    for problem_text in problem_texts:
        # Format using the tokenizer's chat template directly
        prompt = tokenizer.apply_chat_template(
            [{"role": "user", "content": problem_text}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=True,
        )
        prompts.append(prompt)

In [13]:
def detect_category(prompt: str) -> str:
    if "secret bit manipulation rule transforms 8-bit binary numbers" in prompt:
        return "bit_manipulation"
    if "secret encryption rules are used on text" in prompt:
        return "cipher"
    if "secret set of transformation rules is applied to equations" in prompt:
        after_header = prompt.split("Below are a few examples:\n", 1)[1]
        examples_text, rest = after_header.split("\nNow, determine the result for: ", 1)
        question_text = rest.strip()
        if any(c.isdigit() for c in examples_text):
            q_match = re.fullmatch(r"(\d+)(\D)(\d+)", question_text)
            if q_match and re.search(
                r"\d" + re.escape(q_match.group(2)) + r"\d", examples_text
            ):
                return "equation_numeric_deduce"
            return "equation_numeric_guess"
        if len(question_text) == 5:
            q_op = question_text[2]
            for ex_line in examples_text.strip().splitlines():
                inp = ex_line.split(" = ")[0].strip()
                if len(inp) == 5 and inp[2] == q_op:
                    return "cryptarithm_deduce"
        return "cryptarithm_guess"
    if "gravitational constant has been secretly changed" in prompt:
        return "gravity"
    if "converted into a different numeral system" in prompt:
        return "numeral"
    if "secret unit conversion is applied to measurements" in prompt:
        return "unit_conversion"
    raise ValueError("unknown")

In [14]:
# Generate predictions using continuous batching
lora_path = "/kaggle/working"

if RUN_EVALUATION:
    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=LoRARequest("adapter", 1, lora_path),
    )

Rendering prompts:   0%|          | 0/950 [00:00<?, ?it/s]

WARNING 04-12 19:15:17 [input_processor.py:168] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/950 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

# Produce submission

In [15]:
import zipfile as _zf

print(os.listdir("."))
with _zf.ZipFile("submission.zip", "w", _zf.ZIP_DEFLATED) as zf:
    for file in os.listdir("."):
        if "adapter" not in file:
            continue
        if not os.path.isfile(file):
            continue
        zf.write(file)
        os.remove(file)

['README.md', '__notebook__.ipynb', 'checkpoint_complete', 'adapter_model.safetensors', 'adapter_config.json']


In [16]:
print(os.listdir("."))

['README.md', '__notebook__.ipynb', 'checkpoint_complete', 'submission.zip']


# Calculate statistics

In [17]:
if RUN_EVALUATION:
    df["output"] = [output.outputs[0].text for output in outputs]
    df["category"] = [detect_category(problem_text) for problem_text in problem_texts]
    df["predicted"] = df["output"].apply(extract_final_answer)
    df["correct"] = df.apply(
        lambda row: verify(str(row["answer"]), str(row["predicted"])), axis=1
    )
    df["minlogprob"] = [
        min(
            lp.logprob
            for token_dict in output.outputs[0].logprobs
            for lp in token_dict.values()
        )
        if output.outputs[0].logprobs
        else None
        for output in outputs
    ]

In [18]:
# Save mistakes per category
if RUN_EVALUATION:
    os.makedirs("mistakes", exist_ok=True)
    for category in df["category"].unique():
        cat_mistakes = df[(df["category"] == category) & (~df["correct"])]
        if not cat_mistakes.empty:
            cat_mistakes.to_csv(f"mistakes/{category}.csv", index=False)
    
    # Print results table
    stats = df.groupby("category")["correct"].agg(correct="sum", total="count").sort_index()
    stats["correct"] = stats["correct"].astype("int")
    grand_total = stats["total"].sum()
    stats["weightage"] = (stats["total"] / grand_total * 100).map("{:.1f}%".format)
    stats["percentage"] = (stats["correct"] / stats["total"] * 100).map("{:.1f}%".format)
    stats["contribution"] = (stats["correct"] / grand_total * 100).map("{:.1f}%".format)
    overall_pct = stats["correct"].sum() / grand_total * 100
    totals = pd.DataFrame(
        {
            "correct": [stats["correct"].sum()],
            "total": [grand_total],
            "weightage": ["100.0%"],
            "percentage": [f"{overall_pct:.1f}%"],
            "contribution": [f"{overall_pct:.1f}%"],
        },
        index=["TOTAL"],
    )
    results = pd.concat([stats, totals])
    print(results.to_string())
    results.to_csv("results.csv")
    df.to_csv("validation.csv", index=False)

                         correct  total weightage percentage contribution
bit_manipulation             149    169     17.8%      88.2%        15.7%
cipher                       158    162     17.1%      97.5%        16.6%
cryptarithm_deduce             6     71      7.5%       8.5%         0.6%
cryptarithm_guess              3     14      1.5%      21.4%         0.3%
equation_numeric_deduce       42     48      5.1%      87.5%         4.4%
equation_numeric_guess         0      7      0.7%       0.0%         0.0%
gravity                      159    159     16.7%     100.0%        16.7%
numeral                      149    149     15.7%     100.0%        15.7%
unit_conversion              171    171     18.0%     100.0%        18.0%
TOTAL                        837    950    100.0%      88.1%        88.1%
